In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import mlflow
from mlflow.models import infer_signature
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe

import warnings
warnings.filterwarnings("ignore")

/home/std-25-353/miniconda3/envs/ml_flow/lib/python3.10/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";"
)

In [4]:
# ========================== Data Splitting ==========================
train, test = train_test_split(data, test_size=0.25, random_state=42)

X = train.drop(['quality'], axis=1).values.astype(np.float32)
y = train['quality'].values.astype(np.float32)

# Split train into train + validation
train_x, valid_x, train_y, valid_y = train_test_split(X, y, test_size=0.20, random_state=42)

test_x = test.drop(['quality'], axis=1).values.astype(np.float32)
test_y = test['quality'].values.astype(np.float32)

In [5]:
# Infer signature for MLflow
signature = infer_signature(train_x, train_y)

print(f"Train shape: {train_x.shape}, Valid shape: {valid_x.shape}")

# ========================== PyTorch Model ==========================
class WineQualityModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    
    def forward(self, x):
        return self.model(x)

Train shape: (2938, 11), Valid shape: (735, 11)


In [6]:
def train_model(params, epochs, train_x, train_y, valid_x, valid_y, device):
    """
    Train the model with given hyperparameters
    """
    # Convert to tensors
    train_dataset = TensorDataset(torch.tensor(train_x), torch.tensor(train_y).unsqueeze(1))
    valid_dataset = TensorDataset(torch.tensor(valid_x), torch.tensor(valid_y).unsqueeze(1))
    
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False)

    # Model, Loss, Optimizer
    model = WineQualityModel(input_dim=train_x.shape[1]).to(device)
    
    criterion = nn.MSELoss()
    optimizer = optim.SGD(
        model.parameters(),
        lr=params["lr"],
        momentum=params["momentum"]
    )

    # Training Loop
    with mlflow.start_run(nested=True):
        mlflow.log_params(params)
        
        for epoch in range(epochs):
            model.train()
            train_loss = 0.0
            for batch_x, batch_y in train_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                
                optimizer.zero_grad()
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()
            
            # Validation
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for batch_x, batch_y in valid_loader:
                    batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                    outputs = model(batch_x)
                    loss = criterion(outputs, batch_y)
                    val_loss += loss.item()
            
            val_rmse = np.sqrt(val_loss / len(valid_loader))
            
            if (epoch + 1) % 10 == 0 or epoch == 0:
                print(f"Epoch {epoch+1}/{epochs} - Val RMSE: {val_rmse:.4f}")

        # Final evaluation
        eval_rmse = val_rmse
        mlflow.log_metric("eval_rmse", eval_rmse)
        
        # Log the model
        mlflow.pytorch.log_model(model, "model", signature=signature)
        
        return {"loss": eval_rmse, "status": STATUS_OK, "model": model}

In [ ]:
# ========================== Hyperparameter Search Space ==========================
space = {
    "lr": hp.loguniform("lr", np.log(1e-5), np.log(1e-1)),
    "momentum": hp.uniform("momentum", 0.0, 1.0)
}


# ========================== Objective Function for Hyperopt ==========================
def objective(params):
    # You can change epochs as needed
    epochs = 50
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    result = train_model(params, epochs, train_x, train_y, valid_x, valid_y, device)
    return result

In [ ]:
# ========================== Run Hyperparameter Tuning ==========================
mlflow.set_experiment("wine-quality-pytorch")

with mlflow.start_run(run_name="hyperopt_tuning"):
    trials = Trials()
    
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=8,           # Increased a bit from 4
        trials=trials
    )
    
    # Get best trial
    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]
    
    mlflow.log_params(best)
    mlflow.log_metric("best_eval_rmse", best_run["loss"])
    
    print("\n" + "="*50)
    print(f"Best parameters: {best}")
    print(f"Best eval RMSE: {best_run['loss']:.4f}")
    print("="*50)